# 04 — Round 1 tuning (length, LR, max_steps)

**Summary:** Loads the train/val pool from notebook **03** (same **`WORKFLOW_ROOT`** / `RUN_PROFILE_ID`). Sweeps `max_seq_length`, `learning_rate`, and `max_steps`; early stopping on **`eval_loss` only**; each run registers under `<WORKFLOW_ROOT>/models/lora_model_id_<id>/` and appends `<WORKFLOW_ROOT>/lora_tuning_workflow/round1_results.csv`.

**Prerequisites:** Notebook **03** with matching `RUN_PROFILE_ID`. Notebooks **01** and **02** if you use ranked source data.

**Training subset:** `TRAIN_FIRST_N` — if set, uses only the **first N rows of the train split** (after pool load); train/val split uses `shuffle=False` so order matches the pool CSV. If `None`, the full train split is used with shuffled train/val (default behavior).

**Parameters:** `RUN_PROFILE_ID`, `WORKFLOW_ROOT`, `VAL_FRAC`, `TRAIN_FIRST_N`, grids, `ROUND1_BASE_CONFIG`.

**Recommended run order:** 03 → 04 → 05 → **08** (holdout eval for round1–4 models) → 06 → 07 → **09** (holdout eval for `best_extended` / `curriculum`) → 10–12.

#### Colab: mount Google Drive (optional)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#### Install dependencies

In [ ]:
!pip -q install transformers peft accelerate bitsandbytes datasets trl cairosvg pillow scikit-learn lxml pandas

#### Imports, paths

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

PROJECT_DIR = Path('/content/drive/MyDrive/DL_Midterm_Spring_2026_2/svg_project_DL')
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

RUN_PROFILE_ID = 'run_1' #'default'
WORKFLOW_ROOT = PROJECT_DIR / 'outputs' / 'workflow_runs' / RUN_PROFILE_ID
MODELS_ROOT = WORKFLOW_ROOT / 'models'
EXPERIMENT_ROOT = WORKFLOW_ROOT / 'lora_tuning_workflow'
ROUND1_RESULTS_PATH = EXPERIMENT_ROOT / 'round1_results.csv'

print('PROJECT_DIR:', PROJECT_DIR)
print('WORKFLOW_ROOT:', WORKFLOW_ROOT)
print('cuda:', torch.cuda.is_available())

PROJECT_DIR: /content/drive/MyDrive/DL_Midterm_Spring_2026_2/svg_project_DL
WORKFLOW_ROOT: /content/drive/MyDrive/DL_Midterm_Spring_2026_2/svg_project_DL/outputs/workflow_runs/run_1
cuda: True


#### Parameters

In [ ]:
SEED = 42
VAL_FRAC = 0.10
# First N rows of the train split only (None = full train). When set, train/val split is sequential (shuffle=False).
TRAIN_FIRST_N = 7500 #None  # e.g. 400

MODEL_ID = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'

MAX_SEQ_LENGTH_GRID = [1024, 1536]
LEARNING_RATE_GRID = [1e-4, 2e-4]
MAX_STEPS_GRID = [175, 350]

ROUND1_BASE_CONFIG = {
    'lora_r': 16,
    'lora_alpha': 32,
    'lora_dropout': 0.05,
    'per_device_train_batch_size': 4,
    'gradient_accumulation_steps': 8,
    'warmup_ratio': 0.05,
    'lr_scheduler_type': 'cosine',
    'early_stopping_patience': 3,
    'early_stopping_min_delta': 0.0,
    'eval_steps': 50,
}

EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)
print('VAL_FRAC:', VAL_FRAC, '| TRAIN_FIRST_N:', TRAIN_FIRST_N)

VAL_FRAC: 0.1 | TRAIN_FIRST_N: 7500


#### Load pool, train/val split

In [ ]:
from src.core.dataframe import choose_first_existing, train_val_split_df
from src.core.modeling_splits import load_train_val_pool

train_val_pool = load_train_val_pool(WORKFLOW_ROOT)
PROMPT_COL = choose_first_existing(train_val_pool, ['prompt', 'description', 'text'], 'pool')
SVG_COL = choose_first_existing(train_val_pool, ['svg', 'svg_code', 'target', 'label'], 'pool')
_shuffle = TRAIN_FIRST_N is None
train_df, val_df = train_val_split_df(
    train_val_pool, val_frac=VAL_FRAC, seed=SEED, shuffle=_shuffle
)
if TRAIN_FIRST_N is not None:
    train_df = train_df.iloc[: int(TRAIN_FIRST_N)].reset_index(drop=True)
    n_val = min(len(val_df), max(1, int(VAL_FRAC * TRAIN_FIRST_N)))
    val_df = val_df.sample(n=n_val, random_state=SEED).reset_index(drop=True)
print('Train:', len(train_df), 'Val:', len(val_df))

Train: 7500 Val: 4990


#### Round 1 grid (register each run)

In [ ]:
from src.core.runtime import cleanup_memory, set_seed
from src.training.lora.experiments import run_single_experiment_eval_loss_early_stop
from src.training.lora.tuning_utils import append_round_results_csv

set_seed(SEED)
cleanup_memory()

if ROUND1_RESULTS_PATH.exists():
    ROUND1_RESULTS_PATH.unlink()

run_idx = 0
for max_seq_length in MAX_SEQ_LENGTH_GRID:
    for lr in LEARNING_RATE_GRID:
        for max_steps in MAX_STEPS_GRID:
            run_idx += 1
            cfg = {**ROUND1_BASE_CONFIG, 'learning_rate': lr, 'max_steps': int(max_steps)}
            run_name = f'r1_{run_idx:03d}_msl{max_seq_length}_lr{lr}_steps{max_steps}_seed{SEED}'
            print('===', run_name)
            summary, _, run_dir, reg_id = run_single_experiment_eval_loss_early_stop(
                run_name=run_name,
                config=cfg,
                train_df=train_df,
                val_df=val_df,
                prompt_col=PROMPT_COL,
                svg_col=SVG_COL,
                model_id=MODEL_ID,
                max_seq_length=max_seq_length,
                root_dir=EXPERIMENT_ROOT,
                project_root=PROJECT_DIR,
                tuning_stage='round1',
                curriculum=False,
                seed=SEED,
                eval_steps=cfg.get('eval_steps'),
                notes='round1 grid',
                models_root=MODELS_ROOT,
            )
            row = {**summary, 'max_seq_length': max_seq_length, 'tuning_stage': 'round1'}
            append_round_results_csv(ROUND1_RESULTS_PATH, row)
            cleanup_memory()

r1 = pd.read_csv(ROUND1_RESULTS_PATH)
print('Round 1 done. Rows:', len(r1))
# Inspection only — does not select models for later rounds (see notebook 05).
display(r1.sort_values('eval_loss').head(10))

=== r1_001_msl1024_lr0.0001_steps175_seed42


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Adding EOS to eval dataset:   0%|          | 0/4990 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/4990 [00:00<?, ? examples/s]

Step,Training Loss,Validation Loss
50,0.448830,0.431798
100,0.385170,0.394905
150,0.372109,0.377814
200,0.367188,0.367736
250,0.367355,0.362333
300,0.348636,0.360077
350,0.343709,0.359888


Round 1 done. Rows: 8


,run_name,train_rows,val_rows,learning_rate,lora_r,lora_alpha,lora_dropout,per_device_train_batch_size,gradient_accumulation_steps,effective_batch_size,...,best_loss_checkpoint,svg_open_rate,svg_close_rate,xml_parse_rate,render_rate,avg_pred_char_len,submission_valid_rate,registry_model_id,max_seq_length,tuning_stage
7,r1_008_msl1536_lr0.0002_steps350_seed42,7500,4990,0.0002,16,32,0.05,4,8,32,...,/content/drive/MyDrive/DL_Midterm_Spring_2026_...,NaN,NaN,NaN,NaN,NaN,NaN,f590f8aefc614608,1536,round1
3,r1_004_msl1024_lr0.0002_steps350_seed42,7500,4990,0.0002,16,32,0.05,4,8,32,...,/content/drive/MyDrive/DL_Midterm_Spring_2026_...,NaN,NaN,NaN,NaN,NaN,NaN,afd55604e6b647cb,1024,round1
5,r1_006_msl1536_lr0.0001_steps350_seed42,7500,4990,0.0001,16,32,0.05,4,8,32,...,/content/drive/MyDrive/DL_Midterm_Spring_2026_...,NaN,NaN,NaN,NaN,NaN,NaN,a3b2c51c4f544237,1536,round1
6,r1_007_msl1536_lr0.0002_steps175_seed42,7500,4990,0.0002,16,32,0.05,4,8,32,...,/content/drive/MyDrive/DL_Midterm_Spring_2026_...,NaN,NaN,NaN,NaN,NaN,NaN,1c3f07c240d04d1c,1536,round1
1,r1_002_msl1024_lr0.0001_steps350_seed42,7500,4990,0.0001,16,32,0.05,4,8,32,...,/content/drive/MyDrive/DL_Midterm_Spring_2026_...,NaN,NaN,NaN,NaN,NaN,NaN,631bfea447974924,1024,round1
2,r1_003_msl1024_lr0.0002_steps175_seed42,7500,4990,0.0002,16,32,0.05,4,8,32,...,/content/drive/MyDrive/DL_Midterm_Spring_2026_...,NaN,NaN,NaN,NaN,NaN,NaN,f9cfe3de45914b51,1024,round1
4,r1_005_msl1536_lr0.0001_steps175_seed42,7500,4990,0.0001,16,32,0.05,4,8,32,...,/content/drive/MyDrive/DL_Midterm_Spring_2026_...,NaN,NaN,NaN,NaN,NaN,NaN,35aff66e4a4d49b7,1536,round1
0,r1_001_msl1024_lr0.0001_steps175_seed42,7500,4990,0.0001,16,32,0.05,4,8,32,...,/content/drive/MyDrive/DL_Midterm_Spring_2026_...,NaN,NaN,NaN,NaN,NaN,NaN,c00eed730559486c,1024,round1
